Importation des bibliothèque

In [2]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt 
from biopandas.pdb import PandasPdb
import nglview as nv

In [10]:
def view_structure(file_pdb):
    # 1. Charger le fichier généré
    view = nv.show_file(file_pdb)
    # 2. Vider les représentations par défaut
    view.clear_representations()
    # 3. Afficher toute la molécule de manière fine
    view.add_representation('licorice', selection='all', color='element')
    # 4. Mettre en surbrillance spéciale les atomes de Phosphore (les "noeuds" des liaisons phosphodiester)
    view.add_representation('spacefill', selection='_P', color='orange', radius=0.8)
    # 5. Rajouter un "tube" qui suit et met en évidence de façon continue le squelette phosphodiester (backbone)
    view.add_representation('tube', selection='backbone', color='red', radius=0.2)
    
    # # 6. Ajout d'un dégradé de couleur pour voir l'orientation 5' -> 3'
    # # Le schéma 'resindex' colore du bleu (début/5') au rouge (fin/3')
    view.add_representation('cartoon', selection='nucleic', color='resindex')
    
    # 7. Centrer la vue
    view.center()
    # Afficher l'interface
    return view

In [16]:
view_structure("fichier_arn/initial_optimized.pdb")

NGLWidget()

In [6]:
ppdb_df =  PandasPdb().read_pdb("fichier_arn/initial_optimized.pdb")
atom_df = ppdb_df.df["ATOM"]
atom_df.head(20)

,record_name,atom_number,blank_1,atom_name,alt_loc,residue_name,blank_2,chain_id,residue_number,insertion,...,x_coord,y_coord,z_coord,occupancy,b_factor,blank_4,segment_id,element_symbol,charge,line_idx
0,ATOM,1,,P,,G,,A,1,,...,17.035,30.701,-10.063,1.0,0.0,,,P,NaN,0
1,ATOM,2,,OP1,,G,,A,1,,...,15.700,30.562,-10.686,1.0,0.0,,,O,NaN,1
2,ATOM,3,,OP2,,G,,A,1,,...,17.279,31.914,-9.250,1.0,0.0,,,O,NaN,2
3,ATOM,4,,O5',,G,,A,1,,...,17.259,29.356,-9.227,1.0,0.0,,,O,NaN,3
4,ATOM,5,,C5',,G,,A,1,,...,17.574,28.140,-9.932,1.0,0.0,,,C,NaN,4
5,ATOM,6,,C4',,G,,A,1,,...,17.743,26.997,-8.946,1.0,0.0,,,C,NaN,5
6,ATOM,7,,O4',,G,,A,1,,...,19.122,26.985,-8.467,1.0,0.0,,,O,NaN,6
7,ATOM,8,,C1',,G,,A,1,,...,19.181,27.480,-7.137,1.0,0.0,,,C,NaN,7
8,ATOM,9,,N9,,G,,A,1,,...,20.203,28.563,-7.092,1.0,0.0,,,N,NaN,8
9,ATOM,10,,C8,,G,,A,1,,...,20.083,29.891,-7.450,1.0,0.0,,,C,NaN,9


In [10]:
from Bio.PDB import Structure, Model, Chain, Residue, Atom, PDBIO

def generer_structure_depart_c3(seq, distance_moy=5.8, output="initial_c3.pdb"):
    # Création de la hiérarchie
    structure = Structure.Structure("RNA")
    model = Model.Model(0)
    chain = Chain.Chain("A")
    
    for i, nt in enumerate(seq):
        # Création du résidu (nucléotide)
        res = Residue.Residue((" ", i+1, " "), nt, " ")
        # Création de l'atome C3' à sa position x
        atom = Atom.Atom("C3'", [i * distance_moy, 0, 0], 0, 0, " ", "C3'", i+1, "C")
        res.add(atom)
        chain.add(res)
        
    model.add(chain)
    structure.add(model)
    
    # Sauvegarde
    io = PDBIO()
    io.set_structure(structure)
    io.save(output)


def voir_configuration_c3(fichier_pdb):
    """
    Visualisation NGL adaptée au modèle C3'-seulement.
    """
    view = nv.show_file(fichier_pdb)
    view.clear_representations()
    view.add_representation("ball+stick", selection="all", color="cyan", radius=0.25)
    view.center()
    return view


# Exemple rapide :
generer_structure_depart_c3(seq="ACGU"*30, distance_moy=5.8)
voir_configuration_c3("fichier_arn/initial_c3_beads.pdb")

NGLWidget()